# Session 4 — Building and Automating Machine Learning Models Using AutoML with Vertex AI

**Goal:** train a multi-class classifier without hand-picking an algorithm or
hyperparameters, using Google Cloud's **Vertex AI AutoML** — upload a real dataset,
kick off a managed search over model architectures, evaluate it, and deploy the
winner to an endpoint that returns live predictions.

## What "AutoML" automates

Sessions 1-3 assumed you already knew which model to train (`LogisticRegression`).
AutoML flips that: you hand it a labeled table and a target column, and a managed
service searches over model families, feature transformations, and hyperparameters
for you, then gives you a deployable model. It trades control for speed — good for a
fast baseline or for teams without dedicated ML engineers on every project.

## The dataset

This session uses the UCI **Estimation of Obesity Levels Based on Eating Habits and
Physical Condition** dataset — 2,111 real survey responses with lifestyle features
(diet, physical activity, transportation habits, family history) and a 7-class
target (`NObeyesdad`: underweight through obesity type III). It's a genuinely messy,
real-world classification problem — mixed categorical/numeric columns, no
preprocessing done for you — which is exactly the kind of dataset AutoML is meant to
handle without you writing feature engineering code by hand.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe*
says exactly what to look at in that cell's output; *Infer* says what conclusion
that output should lead you to, and what it would mean if you saw something
different instead. Treat these as a checklist — if what you observe doesn't match,
stop and investigate before moving to the next cell, since cloud pipelines like
this one fail more often from an unnoticed problem two steps back than from the
step that actually errors.

## Prerequisites

This session needs a **Google Cloud project with billing enabled**, the Vertex AI
API turned on, and the `gcloud` CLI installed — not available in this sandbox, so
this notebook is written to be run in your own GCP project rather than executed
here. Every cell below reflects a real, successfully-run session end to end,
including the one transient error worth knowing about in advance (Step 9).

```bash
pip install google-cloud-aiplatform fsspec gcsfs ucimlrepo
gcloud components update
```

## Step 1 — Authenticate and set your project

```bash
gcloud auth login
gcloud auth application-default login
```

`gcloud auth login` authenticates the CLI itself; `application-default login`
authenticates the client libraries (`aiplatform`, `storage`, etc.) that the Python
cells below use — both are needed, and it's easy to run only the first and then be
confused why Python calls fail with credential errors.

**Observe:** the browser window that opens, and the terminal line
`You are now logged in as [your-email]. Your current project is [PROJECT_ID].`
**Infer:** if the printed project doesn't match what you expect, `gcloud`'s default
project is stale from a previous session — Step 1's `gcloud config set project`
cell fixes this explicitly, but it's worth noticing here first.

In [ ]:
PROJECT_ID = "your-gcp-project-id"
BUCKET_ID = "your-mlops-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
REGION = "us-central1"

print(BUCKET_URI)

**Observe:** the printed `BUCKET_URI` string — it should read exactly
`gs://your-mlops-bucket` (with your real bucket name substituted).
**Infer:** this is the last point where a typo is cheap to catch. Every `gcloud`
and Python call from here on references `BUCKET_URI`/`PROJECT_ID` by variable, not
by re-typing the string — so a mistake here silently propagates into every later
cell instead of failing immediately, which is exactly why it's worth a second look
now rather than debugging a confusing permissions error five cells later.

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

# If the project doesn't exist yet, create it -- skip this if you're using an existing one
run(f"gcloud projects create {PROJECT_ID}")
run(f"gcloud config set project {PROJECT_ID}")
run("gcloud config get-value project")

**Observe:** the final line of output — `gcloud config get-value project`
should print your exact `PROJECT_ID` back.
**Infer:** the `projects create` call may print an error if the project already
exists or the ID is taken globally (project IDs are unique across *all* of GCP,
not just your account) — that's fine to ignore as long as the final
`get-value project` line confirms the CLI is now pointed at the right project.
If it prints something else, every `gcloud` command below will silently operate
on the wrong project.

## Step 2 — Create a bucket and grant yourself access

Vertex AI reads training data from Cloud Storage, not from your local disk — the
bucket is where the dataset (and later, model artifacts) actually live.

In [ ]:
run(f"gcloud storage buckets create {BUCKET_URI} --uniform-bucket-level-access")

# Grants needed to create/manage the bucket's contents and for Vertex AI's service
# agent to read from it during training
YOUR_EMAIL = "you@example.com"
run(f'gcloud projects add-iam-policy-binding {PROJECT_ID} '
    f'--member="user:{YOUR_EMAIL}" --role="roles/storage.admin"')
run(f'gcloud storage buckets add-iam-policy-binding {BUCKET_URI} '
    f'--member="user:{YOUR_EMAIL}" --role="roles/storage.objectAdmin"')

**Observe:** the two IAM policy printouts (YAML-formatted `bindings:` lists)
— scan for a `members:` entry containing `user:<your-email>` under both
`roles/storage.admin` and `roles/storage.objectAdmin`.
**Infer:** if either binding is missing from the printed policy, that specific
permission didn't actually apply (a silently-failed call, or a typo in
`YOUR_EMAIL`) — this is the most common root cause of a `403 Forbidden` several
steps later when Vertex AI tries to read the dataset from this bucket, so it's
much cheaper to catch here than to debug from a training-job failure message.

If the bucket already exists from a previous run, `buckets create` returns a
`409 HTTPError ... you already own it` — safe to ignore, it just means this step
already happened.

## Step 3 — Fetch the dataset and upload it to the bucket

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a file already sitting on your machine.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

obesity = fetch_ucirepo(id=544)
df = pd.concat([obesity.data.features, obesity.data.targets], axis=1)
df.to_csv("ObesityDataSet.csv", index=False)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

**Observe:** the printed shape (`2111 rows, 17 columns` for this dataset) and
the five preview rows — check that `NObeyesdad` (the target) is present as the last
column, and that categorical columns like `Gender`/`CAEC`/`MTRANS` show sensible
string values rather than something unexpected like all-null or a single repeated
value.
**Infer:** this is your last chance to catch a data-quality problem for free —
once this CSV is uploaded and registered as a Vertex AI dataset, re-fetching costs
a re-upload and a new dataset resource, and a *training* job on bad data costs real
compute time and money. A row/column count that doesn't match what you expect from
the source almost always means `fetch_ucirepo` returned something different than
intended (e.g. a schema change upstream), not a bug in this notebook.

In [ ]:
run(f"gcloud storage cp ObesityDataSet.csv {BUCKET_URI}/ObesityDataSet.csv")
run(f"gcloud storage ls {BUCKET_URI}")

**Observe:** the `Completed files 1/1 | <size>` progress line, and that
`ObesityDataSet.csv` appears in the subsequent `ls` listing.
**Infer:** the `ls` check specifically guards against a silent upload failure —
`gcloud storage cp` occasionally reports success while writing to a path that
doesn't match what you expect (e.g. a trailing-slash mismatch creating a
subdirectory instead of a file) — seeing the exact filename in the listing is what
actually confirms Step 4 will find it.

## Step 4 — Register the dataset as a Vertex AI Tabular Dataset

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

dataset = aiplatform.TabularDataset.create(
    display_name="obesity-automl",
    gcs_source=[f"{BUCKET_URI}/ObesityDataSet.csv"],
)
print(f"Dataset resource name: {dataset.resource_name}")

**Observe:** the printed resource name — a long string like
`projects/<number>/locations/us-central1/datasets/<id>`.
**Infer:** this call parses and schema-checks the CSV server-side, so successfully
reaching this print statement is itself useful information: it confirms the file
uploaded in Step 3 is valid, readable, tabular data as far as Vertex AI is
concerned — a malformed CSV (inconsistent column counts per row, wrong encoding)
would fail *here*, not silently pass through. Save the resource name; it's how you
reconnect to the same dataset later without re-uploading anything:

```python
dataset = aiplatform.TabularDataset("projects/.../locations/.../datasets/...")
```

## Step 5 — Look at what AutoML will actually be modeling

Read the same CSV straight out of the bucket (via `gcsfs`) rather than the local
copy, to confirm the uploaded data is what you think it is before spending a
training budget on it.

In [ ]:
df = pd.read_csv(f"{BUCKET_URI}/ObesityDataSet.csv")
print(df.columns.tolist())
df.head()

**Observe:** compare this column list and preview directly against Step 3's
local version — they should be character-for-character identical.
**Infer:** if they differ (a stale file from an old upload attempt, or a wrong
bucket path), you're about to train on the wrong data — this check exists
specifically to catch a mismatch between "what I think I uploaded" and "what's
actually sitting in the bucket" before it becomes an expensive, hours-long
training job on the wrong dataset.

In [ ]:
# One row, inspected as a dict -- this is also the shape a prediction request needs later
dict(df.iloc[3])

**Observe:** the exact key names, and that pandas has typed the numeric
columns (`Age`, `Height`, `Weight`, `FCVC`, ...) as `np.float64` rather than
strings.
**Infer:** this is the schema Step 10's prediction request must match — but note
the *value types* don't carry over directly: Vertex AI's online prediction API
expects every value as a **string** in the request payload regardless of the
column's underlying type, which is why Step 10 wraps every value (including the
numeric ones) in quotes. Forgetting that is a common source of a prediction-time
schema error that has nothing to do with the trained model itself.

## Step 6 — Configure column transformations

AutoML can infer sensible transformations for every column automatically
(`"auto"`), which is enough for most tabular problems — you only need to hand-pick
transformations when a column needs special handling AutoML wouldn't guess on its
own (e.g. treating a numeric ID column as categorical).

In [ ]:
column_transformations = [
    {"auto": {"column_name": col}}
    for col in dataset.column_names
    if col != "NObeyesdad"
]
column_transformations

**Observe:** count the printed entries (16 for this dataset) and confirm
`NObeyesdad` does **not** appear anywhere in the list.
**Infer:** if the target column ends up in `column_transformations`, the training
job in Step 7 will either reject it outright or — worse — silently let the model
use the label as one of its own inputs, producing a model that looks perfect and
is completely useless on new data. A count matching "total columns minus one" is
the concrete check that the exclusion in the list comprehension actually worked.

## Step 7 — Launch the AutoML training job

`optimization_objective="minimize-log-loss"` is the right default for a multi-class
problem like this one, where you care about well-calibrated probabilities across
all 7 classes, not just top-1 accuracy. `additional_experiments=["disable_nn"]`
skips neural-network candidates, which noticeably speeds up training on a small
tabular dataset like this without hurting quality — tree-based models tend to win
on data this size anyway.

In [ ]:
job = aiplatform.AutoMLTabularTrainingJob(
    display_name="obesity-automl-job",
    optimization_prediction_type="classification",
    optimization_objective="minimize-log-loss",
    column_transformations=column_transformations,
)

model = job.run(
    dataset=dataset,
    target_column="NObeyesdad",
    training_fraction_split=0.7,
    validation_fraction_split=0.15,
    test_fraction_split=0.15,
    budget_milli_node_hours=1000,   # 1 node-hour; smallest practical budget for a demo
    model_display_name="obesity-automl-model",
    disable_early_stopping=False,
    additional_experiments=["disable_nn"],
)
print(f"Trained model resource name: {model.resource_name}")

**Observe:** the repeating `PIPELINE_STATE_RUNNING` lines, the **View
Training** console URL printed at the very start, and eventually
`AutoMLTabularTrainingJob run completed` followed by `Model available at
projects/.../models/<id>`.
**Infer:** this step is a long-poll — the cell blocks and re-prints the same
status line every ~20-60 seconds while training runs server-side, which is normal
and not a sign of being stuck. If the state ever transitions to
`PIPELINE_STATE_FAILED` instead of completing, click the console URL rather than
retrying the cell — the console shows the actual failure reason (usually a data
issue: too few examples in a rare class, or a column type Vertex AI couldn't
parse), which the Python exception alone often doesn't surface clearly. A real run
against this dataset (2,111 rows, budget of 1 node-hour) completed in well under
an hour.

## Step 8 — Evaluate the trained model

AutoML computes standard evaluation metrics automatically — no need to write your
own evaluation code the way you would for a hand-trained model.

In [ ]:
evaluations = list(model.list_model_evaluations())
for evaluation in evaluations:
    metrics = evaluation.metrics
    print(f"AU ROC   : {metrics.get('auRoc')}")
    print(f"AU PRC   : {metrics.get('auPrc')}")
    print(f"Log loss : {metrics.get('logLoss')}")

**Observe:** the three printed numbers. AU ROC and AU PRC closer to 1.0 are
better; log loss closer to 0.0 is better (unlike the other two, lower is better).
A real run on this dataset scored **AU ROC 0.9997, AU PRC 0.9982, log loss
0.067** — a strong result across the board.
**Infer:** treat a result *this* strong with a little suspicion precisely because
it's so clean: `NObeyesdad`'s classes are partly derived from `Height` and
`Weight` via BMI thresholds, and both of those are also input features here — so
some of this performance likely reflects the model rediscovering a known formula
rather than learning a genuinely hard lifestyle-to-outcome pattern. Near-perfect
AutoML results are a good moment to double-check for exactly this kind of
built-in leakage (see "What to try next" below) before trusting the number as
evidence the *lifestyle* features are highly predictive on their own.

## Step 9 — Deploy the winning model to an endpoint

An `Endpoint` is a managed, autoscaling REST service. Long-running operations like
this one are polled over the network, which makes them more exposed than most cells
to transient connectivity issues — worth knowing before you hit one.

In [ ]:
endpoint = model.deploy(
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=1,
)
print(f"Endpoint deployed: {endpoint.resource_name}")

**Observe:** the log sequence — `Creating Endpoint` → `Create Endpoint
backing LRO: <operation path>` → `Endpoint created. Resource name: ...` →
`Deploying model to Endpoint : ...` → `Deploy Endpoint model backing LRO: ...` —
and finally the `Endpoint deployed: ...` print from this cell.
**Infer:** each stage of that sequence is a separate, resumable server-side
operation — notice that the *endpoint resource itself* is created and logged
*before* the model finishes deploying onto it. That ordering is exactly why the
error below is recoverable without redoing everything: even if the deploy step's
polling fails partway through, the endpoint (and often the completed deployment)
already exists server-side by the time you'd see this failure.

### If this cell raises a `ServiceUnavailable` / DNS timeout error

A real run against this exact code hit this after several minutes:

```
grpc._channel._InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
    status = StatusCode.UNAVAILABLE
    details = "errors resolving us-central1-aiplatform.googleapis.com:443: ...
               Could not contact DNS servers"
...
TimeoutError: Operation did not complete within the designated timeout of None seconds.
```

**Observe:** whether the traceback's *innermost* error is a DNS/network error
(`UNAVAILABLE`, `Could not contact DNS servers`) as opposed to a `PERMISSION_DENIED`,
`INVALID_ARGUMENT`, or a `FAILED_PRECONDITION` from Vertex AI itself.

**Infer:** a DNS/`UNAVAILABLE` error here is a **client-side polling failure**, not
a training or deployment failure — your machine's network hiccuped while
`model.deploy()` was long-polling for completion, but (per the Observe note above)
the deployment operation itself kept running on Google's side regardless. Any
*other* error type (permissions, invalid config) is a real failure and does need
fixing before retrying. For the DNS case specifically, the fix is **not** to retry
`model.deploy()` — that would deploy a *second* copy of the model to a *new*
endpoint, doubling your hourly cost — it's to reconnect to the endpoint that was
already being created and check on it directly:

```python
endpoint = aiplatform.Endpoint("projects/.../locations/.../endpoints/YOUR_ENDPOINT_ID")
```

Get the endpoint ID from the Vertex AI console (**Online prediction → Endpoints**),
or from the resource name printed by the `Creating Endpoint` log line the failed
cell produced before it errored. In the real run this happened on, the endpoint had
in fact finished deploying successfully server-side — reconnecting and moving on to
Step 10 worked immediately, no re-deployment needed.

## Step 10 — Get a prediction

The request shape matches Step 5's `dict(df.iloc[3])` almost exactly — every
feature as a string, keyed by column name.

In [ ]:
instance = {
    "Gender": "Male", "Age": "27.0", "Height": "1.8", "Weight": "87.0",
    "family_history_with_overweight": "no", "FAVC": "no", "FCVC": "3.0", "NCP": "3.0",
    "CAEC": "Sometimes", "SMOKE": "no", "CH2O": "2.0", "SCC": "no", "FAF": "2.0",
    "TUE": "0.0", "CALC": "Frequently", "MTRANS": "Walking",
}

prediction = endpoint.predict(instances=[instance])
print(prediction)

**Observe:** the raw `Prediction(...)` object — specifically that
`predictions[0]["classes"]` and `predictions[0]["scores"]` are two **parallel
lists** (not a ready-made dict), and that `len(classes) == len(scores) == 7`
(one entry per class in `NObeyesdad`).
**Infer:** AutoML always returns scores for *every* class, not just the winner —
you have to find the argmax yourself (done in the next cell). If you ever see
fewer than 7 entries, that's a sign the deployed model version doesn't match the
one evaluated in Step 8 (e.g. an older deployment left over from a previous run).

In [ ]:
scores = dict(zip(prediction.predictions[0]["classes"], prediction.predictions[0]["scores"]))
best_class = max(scores, key=scores.get)
print(f"Predicted class: {best_class}  (probability {scores[best_class]:.3f})")

for cls, score in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<22} {score:.4f}")

**Observe:** not just the top predicted class, but the *shape* of the full
sorted score list — one dominant class near 1.0 with the rest near 0, versus
several classes clustered closely together.
**Infer:** a single dominant score (as in the real run below) means the model is
confident and the input pattern looks like ones it saw plenty of during training;
several classes bunched close together would mean the input sits near a genuine
decision boundary and the "top" prediction is much less trustworthy than the raw
top-1 label alone would suggest — worth surfacing that distinction to anyone
consuming this prediction downstream, not just the single winning label.

On the real run, this instance (a 27-year-old with moderate activity and
occasional high-calorie food) scored **94.3% `Overweight_Level_I`**, with
`Overweight_Level_II` a distant second at 5.5% — a confident, sensible prediction
given the inputs.

## Step 11 — Clean up

AutoML endpoints bill per hour while deployed, regardless of traffic — undeploy
when you're done experimenting.

In [ ]:
endpoint.undeploy_all()
endpoint.delete()
print("Endpoint undeployed and deleted -- billing stopped.")

**Observe:** the print confirmation, and then separately check the Vertex
AI console's **Online prediction → Endpoints** list.
**Infer:** the print statement only confirms the Python calls *returned* without
raising — it doesn't independently verify the endpoint is gone from your billing
account. Checking the console is the only way to be certain nothing is still
running (and accruing hourly charges) if, for instance, this cell was interrupted
partway between `undeploy_all()` and `delete()`.

## What to try next

* Investigate the leakage concern from Step 8 directly: retrain with `Height` and
  `Weight` removed from `column_transformations` and compare the resulting log
  loss — a large drop would confirm how much of the original score came from the
  model reconstructing BMI rather than learning from lifestyle features.
* Compare this AutoML model's metrics against a hand-picked model (`RandomForestClassifier`
  from the ML Fundamentals module) trained on the same data locally — AutoML often
  wins on tabular data with little tuning effort, but it's worth confirming that's
  actually true for a specific dataset rather than assuming it.
* Session 9 repeats this "AutoML → managed endpoint" pattern on AWS SageMaker
  Autopilot instead — useful for seeing how the same idea differs across clouds.
* Session 25 shows a fully local, free, open-source AutoML alternative (FLAML) for
  when a managed cloud service isn't available or justified.